# EDA

Load the three project parquets and produce a per-series mini-table
(inception date, n_obs, % missing, friendly name, source ticker) for
the EDA report. Outputs `data/quality/data_inventory.{csv,md}`.

In [1]:
# === Step 1: Data inventory ===

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

# Detect repo root
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

DATA_DIR = ROOT / "data" / "processed"

PRICES_PATH  = DATA_DIR / "prices.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
EXOG_PATH    = DATA_DIR / "exogenous_features.parquet"
SUMMARY_PATH = DATA_DIR / "summary.csv"

In [3]:
prices  = pd.read_parquet(PRICES_PATH)
exog = pd.read_parquet(EXOG_PATH)
returns = pd.read_parquet(RETURNS_PATH)

# Index sanity
for name, df in [("prices", prices), ("exog", exog), ("returns", returns)]:
    if not isinstance(df.index, pd.DatetimeIndex):
        print(f"WARNING: {name} index is {type(df.index).__name__}, not DatetimeIndex")

def file_overview(name, df):
    return {
        "file":       name,
        "n_rows":     len(df),
        "n_cols":     df.shape[1],
        "first_date": df.index.min().strftime("%Y-%m-%d"),
        "last_date":  df.index.max().strftime("%Y-%m-%d"),
        "n_days":     (df.index.max() - df.index.min()).days,
    }

overview = pd.DataFrame([
    file_overview("prices",  prices),
    file_overview("exog",    exog),
    file_overview("returns", returns),
])
print(overview.to_string(index=False))

   file  n_rows  n_cols first_date  last_date  n_days
 prices    2706      15 2016-01-01 2026-05-15    3787
   exog    2706       7 2016-01-01 2026-05-15    3787
returns    2706      15 2016-01-01 2026-05-15    3787


In [7]:
assert prices.shape[1] == 15, f"prices: expected 15 cols, got {prices.shape[1]}"
assert exog.shape[1] == 7, f"exog: expected 7 cols, got {exog.shape[1]}"
print("Column counts OK (15 prices + 7 exog)\n")

# Print actual column names so you know what to populate in SERIES_INFO
print("prices columns:")
print(list(prices.columns))
print("\nexog columns:")
print(list(exog.columns))

Column counts OK (15 prices + 7 exog)

prices columns:
['WTI', 'Brent', 'RBOB', 'HO', 'HH_NG', 'TTF', 'USO', 'UNG', 'UGA', 'AUDUSD', 'DXY', 'BrentWTI_spread', 'USGCJet', 'JetBrent_crack', 'HOBrent_crack']

exog columns:
['RF', 'TNX', 'OilVol', 'VIX', 'CrudeStocks', 'DistStocks', 'RefineryUtil']


In [17]:
# Update the macro block to match the columns printed above.
SERIES_INFO = {
    # ---- prices: outrights (positive series) ----
    "Brent":   {"name": "Brent crude (front-month)",      "category": "crude"},
    "WTI":     {"name": "WTI crude (front-month)",        "category": "crude"},
    "USGCJet": {"name": "US Gulf Coast jet fuel",         "category": "product"},
    "HO":      {"name": "NY Harbor heating oil",          "category": "product"},
    "RBOB":    {"name": "RBOB gasoline",                  "category": "product"},
    "HH_NG":   {"name": "Henry Hub natural gas",          "category": "gas"},
    "TTF":     {"name": "Dutch TTF natural gas",          "category": "gas"},
    # ---- FX / index ----
    "AUDUSD":  {"name": "AUD/USD exchange rate",          "category": "fx"},
    "DXY":     {"name": "US Dollar Index",                "category": "fx"},
    # ---- ETFs ----
    "USO":     {"name": "United States Oil Fund",         "category": "etf"},
    "UNG":     {"name": "United States Natural Gas Fund", "category": "etf"},
    "UGA":     {"name": "United States Gasoline Fund",    "category": "etf"},
    # ---- derived spreads / cracks ----
    "BrentWTI_spread": {"name": "Brent–WTI spread",         "category": "spread"},
    "HOBrent_crack":   {"name": "Heating oil–Brent crack",  "category": "crack"},
    "JetBrent_crack":  {"name": "Jet fuel–Brent crack",     "category": "crack"},
    # ---- exogenous: physical fundamentals (EIA weekly, forward-filled to daily) ----
    "CrudeStocks":   {"name": "US crude oil inventories (EIA)",     "category": "fundamental"},
    "DistStocks":    {"name": "US distillate inventories (EIA)",    "category": "fundamental"},
    "RefineryUtil":  {"name": "US refinery utilization rate (EIA)", "category": "fundamental"},
    # ---- exogenous: volatility indices ----
    "OilVol": {"name": "CBOE crude oil volatility index (^OVX)",          "category": "vol"},
    "VIX":    {"name": "CBOE volatility index (^VIX)",                    "category": "vol"},
    # ---- exogenous: rates ----
    "TNX":    {"name": "10-year Treasury yield (CBOE ^TNX, % p.a. ×10)",  "category": "rates"},
    "RF":     {"name": "3-month T-bill yield (CBOE ^IRX, % p.a.)",       "category": "rates"},
}

# Coverage check
needed   = set(prices.columns) | set(exog.columns)
covered  = set(SERIES_INFO.keys())
missing  = needed - covered
unused   = covered - needed
if missing:
    print(f"Add SERIES_INFO entries for: {sorted(missing)}")
if unused:
    print(f"SERIES_INFO has entries that aren't in the data: {sorted(unused)}")
if not missing and not unused:
    print("SERIES_INFO covers all 22 columns")

SERIES_INFO covers all 22 columns


In [18]:
def inventory_row(ticker, s, info):
    """One row per series: inception, last obs, n_obs, % missing within window."""
    valid = s.dropna()
    WEEKLY_SERIES = {"CrudeStocks", "DistStocks", "RefineryUtil"}
    if len(valid):
        inception = valid.index.min()
        last_obs  = valid.index.max()
        # % missing measured from inception → last_obs, so late-starting series
        # aren't unfairly penalised for not having pre-history.
        in_window = s.loc[inception:last_obs]
        pct_miss  = 100 * in_window.isna().mean()
    else:
        inception, last_obs, pct_miss = pd.NaT, pd.NaT, np.nan

    meta = info.get(ticker, {})
    return {
        "ticker":        ticker,
        "friendly_name": meta.get("name", "(missing — add to SERIES_INFO)"),
        "category":      meta.get("category", "?"),
        "frequency": "weekly→daily ffill" if ticker in WEEKLY_SERIES else "daily",
        "inception":     inception.strftime("%Y-%m-%d") if pd.notna(inception) else "",
        "last_obs":      last_obs.strftime("%Y-%m-%d")  if pd.notna(last_obs)  else "",
        "n_obs":         int(len(valid)),
        "pct_missing":   round(pct_miss, 2),
    }

rows = []
for col in prices.columns:
    rows.append(inventory_row(col, prices[col], SERIES_INFO))
for col in exog.columns:
    rows.append(inventory_row(col, exog[col], SERIES_INFO))

inventory = (pd.DataFrame(rows)
               .sort_values(["category", "ticker"])
               .reset_index(drop=True))
inventory

,ticker,friendly_name,category,frequency,inception,last_obs,n_obs,pct_missing
0,HOBrent_crack,Heating oil–Brent crack,crack,daily,2016-01-04,2026-05-15,2705,0.0
1,JetBrent_crack,Jet fuel–Brent crack,crack,daily,2016-01-04,2026-05-15,2705,0.0
2,Brent,Brent crude (front-month),crude,daily,2016-01-04,2026-05-15,2705,0.0
3,WTI,WTI crude (front-month),crude,daily,2016-01-04,2026-05-15,2705,0.0
4,UGA,United States Gasoline Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0
5,UNG,United States Natural Gas Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0
6,USO,United States Oil Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0
7,CrudeStocks,US crude oil inventories (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0
8,DistStocks,US distillate inventories (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0
9,RefineryUtil,US refinery utilization rate (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0


In [19]:
summary = pd.read_csv(SUMMARY_PATH)
inventory = inventory.merge(
    summary[["name", "yahoo_symbol"]],
    left_on="ticker", right_on="name", how="left",
).drop(columns="name")

# Derived series + EIA fundamentals aren't in summary.csv — mark explicitly
inventory["yahoo_symbol"] = inventory["yahoo_symbol"].fillna("(derived/EIA)")
inventory

,ticker,friendly_name,category,frequency,inception,last_obs,n_obs,pct_missing,yahoo_symbol
0,HOBrent_crack,Heating oil–Brent crack,crack,daily,2016-01-04,2026-05-15,2705,0.0,(derived/EIA)
1,JetBrent_crack,Jet fuel–Brent crack,crack,daily,2016-01-04,2026-05-15,2705,0.0,(derived/EIA)
2,Brent,Brent crude (front-month),crude,daily,2016-01-04,2026-05-15,2705,0.0,BZ=F
3,WTI,WTI crude (front-month),crude,daily,2016-01-04,2026-05-15,2705,0.0,CL=F
4,UGA,United States Gasoline Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0,UGA
5,UNG,United States Natural Gas Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0,UNG
6,USO,United States Oil Fund,etf,daily,2016-01-04,2026-05-15,2705,0.0,USO
7,CrudeStocks,US crude oil inventories (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0,(derived/EIA)
8,DistStocks,US distillate inventories (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0,(derived/EIA)
9,RefineryUtil,US refinery utilization rate (EIA),fundamental,weekly→daily ffill,2016-01-01,2026-05-15,2706,0.0,(derived/EIA)


In [20]:
# Detect repo root
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

# Output folder
OUT_DIR = ROOT / "data" / "quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save files
inventory.to_csv(OUT_DIR / "data_inventory.csv", index=False)
inventory.to_markdown(OUT_DIR / "data_inventory.md", index=False)

print(f"Wrote {OUT_DIR / 'data_inventory.csv'}")
print(f"Wrote {OUT_DIR / 'data_inventory.md'}")

Wrote c:\Users\Bryant\Documents\USYD\Year 3\sem 2\QBUS3850\group assignment\repo_v2\Time-Series-Forecasting\data\quality\data_inventory.csv
Wrote c:\Users\Bryant\Documents\USYD\Year 3\sem 2\QBUS3850\group assignment\repo_v2\Time-Series-Forecasting\data\quality\data_inventory.md
